# 04 — Validación cruzada 5-fold subject-independent (protocolo del paper)

Este notebook implementa la evaluación **correcta** del dataset UTA-RLDD, siguiendo
el protocolo de Ghoddoosian et al. (2019), *"A Realistic Dataset and Baseline
Temporal Model for Early Drowsiness Detection"*.

**Tarea:** binaria — alerta (0) vs somnoliento (2). Se descarta la clase intermedia
*baja vigilancia* por ambigua.

## 1. Por qué un solo split engaña

En `02_training` y `02b_temporal_model` usamos un único split fijo 33/5/5 sujetos.
Al evaluar sobre el **test set real** (sujetos nunca vistos), todos los modelos colapsan:

| Modelo | Val F1 | **Test F1** |
|---|---|---|
| MobileNetV2 (frame) | 0.65 | **0.51** |
| ResNet50V2 (frame)  | 0.68 | **0.44** |
| Temporal (ResNet+BiGRU) | 0.78 | **0.46** |

**Test ≈ 0.50 = azar.** Verificamos que NO hay data leakage. El problema es el
**protocolo de evaluación**, no los modelos:

1. **Varianza del split**: con solo 43 sujetos, 5 en test es una tirada de dados.
   El val F1 de 0.78 es el pico afortunado de una curva ruidosa sobre 5 sujetos.
2. **Atajos de sesión**: cada sujeto grabó sus 3 videos en sesiones distintas
   (su propio teléfono, distinta iluminación/fondo). El modelo aprende la sesión,
   no la somnolencia.
3. **Etiquetas por video, no por frame**: en el video "somnoliento" hay cientos
   de frames con ojos abiertos idénticos a "alerta".
4. **Ruido por-frame**: una predicción aislada es inestable.

## 2. El protocolo correcto

UTA-RLDD se distribuye en **5 folds a propósito**. El paper reporta:

- **5-fold cross-validation subject-independent**: se entrena 5 veces, cada fold
  deja ~1/5 de sujetos afuera como test. Se promedia → mata la varianza del split.
- **Evaluación a nivel de VIDEO**, no de frame: se agregan las predicciones de
  todos los frames de un video (voto mayoritario) → una predicción por video.
  Esto suaviza el ruido por-frame, que es la causa del test en azar.

**Baseline del paper:** ~0.65 accuracy en la tarea 3-clases con un modelo temporal
sobre features de parpadeo. Los autores del dataset, con modelado temporal real,
llegaron a ~0.65 — ese es el techo realista subject-independent.

> **El entrenamiento pesado corre por terminal** (el kernel de Jupyter se cuelga en WSL2):
> ```bash
> source /home/lilidl/pnl/.venv/bin/activate
> cd /home/lilidl/tp_final_cv2
> python train_kfold.py --arch mobilenetv2   # ~4 h
> python train_kfold.py --arch resnet50v2    # ~4 h
> ```
> Cada corrida guarda `checkpoints/kfold/summary_<arch>.json`. Este notebook los carga y analiza.

## 3. Cargar resultados de los 5 folds

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PAPER_BASELINE = 0.65   # acc 3-clases, Ghoddoosian et al. 2019

def load_summary(arch):
    path = Path(f'../checkpoints/kfold/summary_{arch}.json')
    if not path.exists():
        print(f'[falta] {path} — corré: python train_kfold.py --arch {arch}')
        return None
    with open(path) as f:
        return json.load(f)

summ_mb = load_summary('mobilenetv2')
summ_rn = load_summary('resnet50v2')

## 4. Desglose por fold

In [ ]:
def folds_df(summ):
    if summ is None:
        return None
    rows = [{
        'fold': r['fold'] + 1,
        'val_F1': r['val_f1_best'],
        'frame_F1': r['frame_f1'],
        'video_acc': r['video_accuracy'],
        'video_F1': r['video_f1'],
        'n_videos': r['n_videos'],
    } for r in summ['folds']]
    return pd.DataFrame(rows)

for name, summ in [('MobileNetV2', summ_mb), ('ResNet50V2', summ_rn)]:
    df = folds_df(summ)
    if df is not None:
        print(f'\n=== {name} ===')
        print(df.to_string(index=False))
        print(f"  media: frame_F1={summ['frame_f1_mean']:.4f}±{summ['frame_f1_std']:.4f}  "
              f"video_acc={summ['video_acc_mean']:.4f}±{summ['video_acc_std']:.4f}")

## 5. Frame-level vs Video-level por fold

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, name, summ, color in [(axes[0], 'MobileNetV2', summ_mb, 'steelblue'),
                               (axes[1], 'ResNet50V2',  summ_rn, 'indianred')]:
    if summ is None:
        ax.set_title(f'{name} — sin datos'); continue
    folds = [r['fold'] + 1 for r in summ['folds']]
    frame = [r['frame_f1'] for r in summ['folds']]
    video = [r['video_f1'] for r in summ['folds']]
    x = np.arange(len(folds)); w = 0.35
    ax.bar(x - w/2, frame, w, label='frame F1', color=color, alpha=0.5)
    ax.bar(x + w/2, video, w, label='video F1', color=color)
    ax.axhline(summ['video_f1_mean'], color=color, ls='-',  alpha=0.7,
               label=f"video F1 medio={summ['video_f1_mean']:.3f}")
    ax.axhline(PAPER_BASELINE, color='k', ls='--', alpha=0.6, label=f'paper ~{PAPER_BASELINE}')
    ax.axhline(0.5, color='gray', ls=':', alpha=0.5, label='azar')
    ax.set_xticks(x); ax.set_xticklabels([f'fold {f}' for f in folds])
    ax.set_title(name); ax.set_ylabel('F1-macro'); ax.set_ylim(0, 1)
    ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../docs/kfold_frame_vs_video.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Comparación final vs paper

In [ ]:
rows = []
if summ_mb:
    rows.append(['MobileNetV2', summ_mb['frame_f1_mean'], summ_mb['frame_f1_std'],
                 summ_mb['video_acc_mean'], summ_mb['video_f1_mean'], summ_mb['video_f1_std']])
if summ_rn:
    rows.append(['ResNet50V2', summ_rn['frame_f1_mean'], summ_rn['frame_f1_std'],
                 summ_rn['video_acc_mean'], summ_rn['video_f1_mean'], summ_rn['video_f1_std']])

if rows:
    comp = pd.DataFrame(rows, columns=[
        'modelo', 'frame_F1_mean', 'frame_F1_std',
        'video_acc_mean', 'video_F1_mean', 'video_F1_std'])
    print(comp.to_string(index=False))
    print(f'\nPaper baseline (3-clases, temporal): ~{PAPER_BASELINE} acc')
    print('Nota: nuestra tarea es binaria — esperable >= baseline 3-clases.')
else:
    print('Sin resultados todavía. Corré train_kfold.py para ambas arquitecturas.')

## 7. Conclusiones

- El **single split engaña**: val alto, test en azar. Con 43 sujetos la varianza
  de un split fijo es demasiado grande para ser una métrica confiable.
- El **5-fold CV subject-independent** da un estimado estable (media ± std) — es
  el protocolo con el que el dataset fue diseñado para evaluarse.
- La **agregación a nivel de video** sube y estabiliza el número al suavizar el
  ruido de etiquetas por-frame, y es la métrica que reporta el paper.
- **0.80 subject-independent no es realista** con crops faciales crudos: los
  autores del dataset llegaron a ~0.65 con modelado temporal del parpadeo real.
  El valor de este TP está en demostrar **por qué**, con evidencia.